# SVC RBF — aggressive focused search (Colab)

**Run all. SVC only.** This version pushes the promising result further while fixing an important evaluation issue: feature ranking is performed **inside each CV training fold**, never on the held-out engines. This makes the reported score leakage-safe. `test.csv` is not read anywhere in this notebook.


In [ ]:
!pip -q install scikit-learn pandas numpy matplotlib
from pathlib import Path
import time, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score
SEED=42
N_SPLITS=5
C_VALUES=[0.75,1.0,1.25,1.5,1.75,2.0,2.5,3.0,4.0,5.0]
GAMMA_VALUES=[0.007,0.01,0.0125,0.015,0.0175,0.02,0.025,0.03,0.04,0.05]
print(f'Focused SVC grid: {len(C_VALUES)*len(GAMMA_VALUES)} combinations')

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e: print('Drive mount unavailable:',e)
candidates=[Path('/content/data'),Path('/content/drive/MyDrive/hackathon/data'),Path('/content/drive/MyDrive/data')]
DATA_DIR=next((p for p in candidates if (p/'train.csv').exists() and (p/'val.csv').exists()),None)
if DATA_DIR is None: raise FileNotFoundError('Put train.csv and val.csv in MyDrive/hackathon/data/ or /content/data/')
train=pd.read_csv(DATA_DIR/'train.csv'); val=pd.read_csv(DATA_DIR/'val.csv')
assert 'label' not in train.columns
assert {'label','engine_id'}.issubset(val.columns)
assert set(train.engine_id).isdisjoint(set(val.engine_id))
print('DATA_DIR:',DATA_DIR); print('train:',train.shape,'engines=',train.engine_id.nunique()); print('val:',val.shape,'engines=',val.engine_id.nunique())

In [ ]:
freq=sorted([c for c in train.columns if c.startswith('mV_')],key=lambda x:int(x.split('_')[1]))
def features(df):
    x=df[freq].copy().interpolate(axis=1,limit_direction='both').fillna(0); a=x.to_numpy(np.float32); d=np.diff(a,axis=1)
    out=np.column_stack([a,a.mean(1),a.std(1),a.min(1),a.max(1),np.ptp(a,axis=1),np.mean(a*a,1),np.mean(np.abs(a),1),a.argmax(1),a.argmin(1),d.mean(1),d.std(1),d.min(1),d.max(1),a[:,:7].mean(1),a[:,7:14].mean(1),a[:,14:].mean(1)])
    names=freq+['mean','std','min','max','range','energy','abs_mean','argmax','argmin','diff_mean','diff_std','diff_min','diff_max','low_mean','mid_mean','high_mean']
    return pd.DataFrame(out,columns=names,index=df.index).replace([np.inf,-np.inf],np.nan).fillna(0)
X=features(val); names=X.columns.tolist(); y=val.label.astype(str).to_numpy(); groups=val.engine_id.to_numpy()
print('Features:',len(names),'raw spectrum:',len(freq))

## Why this version is more trustworthy

The previous `0.8446` search ranked features using **all labeled validation engines before GroupKFold**. That leaks information from the held-out fold into feature selection. The new notebook ranks features separately on each training fold. The score may therefore move up or down, but it is a much better estimate of true generalization. We still search only SVC RBF.

In [ ]:
def rank_features(Xtr,ytr,top_k):
    r=ExtraTreesClassifier(n_estimators=400,max_features=1.0,max_depth=15,min_samples_leaf=2,class_weight='balanced',n_jobs=-1,random_state=SEED)
    r.fit(Xtr,ytr)
    imp=pd.Series(r.feature_importances_,index=Xtr.columns).sort_values(ascending=False)
    return imp.head(min(top_k,len(imp))).index.tolist()

# We compare raw spectrum against nested top-k feature selection.
FEATURE_MODES=['raw_21','top_12','top_16','top_20','top_24','top_28','all']
print('Feature modes:',FEATURE_MODES)

In [ ]:
grid=list(ParameterGrid({'C':C_VALUES,'gamma':GAMMA_VALUES}))
gkf=GroupKFold(n_splits=N_SPLITS)
def macro(y_true,pred): return f1_score(y_true,pred,average='macro',zero_division=0)

# Precompute leakage-safe feature selections once per fold; they depend only on that fold's training data.
fold_info=[]
for fi,(tr,ho) in enumerate(gkf.split(X,groups=groups),1):
    raw=[c for c in freq if c in names]
    top12=rank_features(X.iloc[tr],y[tr],12); top16=rank_features(X.iloc[tr],y[tr],16); top20=rank_features(X.iloc[tr],y[tr],20); top24=rank_features(X.iloc[tr],y[tr],24); top28=rank_features(X.iloc[tr],y[tr],28)
    fold_info.append({'tr':tr,'ho':ho,'sets':{'raw_21':raw,'top_12':top12,'top_16':top16,'top_20':top20,'top_24':top24,'top_28':top28,'all':names}})
    print(f'fold {fi}: top24={top24[:8]} ...')

rows=[]; start=time.time(); total=len(FEATURE_MODES)*len(grid); done=0
for fs_name in FEATURE_MODES:
    print(f'\n=== {fs_name} ===')
    for pi,p in enumerate(grid,1):
        fold_scores=[]
        for fi,info in enumerate(fold_info,1):
            fs=info['sets'][fs_name]; tr,ho=info['tr'],info['ho']
            m=Pipeline([('scale',StandardScaler()),('svc',SVC(kernel='rbf',class_weight='balanced',random_state=SEED,cache_size=1024,**p))])
            m.fit(X.iloc[tr][fs],y[tr]); pred=m.predict(X.iloc[ho][fs]); fold_scores.append(macro(y[ho],pred))
        row={'feature_set':fs_name,'n_features':len(info['sets'][fs_name]),'C':p['C'],'gamma':p['gamma'],'raw_score':float(np.mean(fold_scores)),'std':float(np.std(fold_scores)),'min_fold':float(np.min(fold_scores))}
        rows.append(row); done+=1
        if pi in (1,10,20,30,40,50,60,70,80,90,100): print(f'{pi:03d}/{len(grid)} score={row["raw_score"]:.4f} std={row["std"]:.4f} C={p["C"]} gamma={p["gamma"]}')
results=pd.DataFrame(rows).sort_values(['raw_score','min_fold'],ascending=False).reset_index(drop=True)
results.to_csv('svc_aggressive_search.csv',index=False)
print(f'\nElapsed: {(time.time()-start)/60:.1f} min')
display(results.head(25))

In [ ]:
best=results.iloc[0]
print('=== BEST SVC ==='); print(best.to_string())
print('\n=== BEST PER FEATURE SET ===')
display(results.sort_values(['raw_score','min_fold'],ascending=False).groupby('feature_set',as_index=False).first().sort_values('raw_score',ascending=False))
print('\nInterpretation: prioritize high raw_score, but use std/min_fold to detect unstable winners.')
print('No pseudo-labeling. No test.csv. No final submission is generated by this notebook.')

In [ ]:
# Compact heatmap of the best feature set.
fs=str(best.feature_set); h=results[results.feature_set==fs].pivot(index='gamma',columns='C',values='raw_score').sort_index()
plt.figure(figsize=(11,5)); plt.imshow(h.values,aspect='auto',origin='lower'); plt.xticks(range(len(h.columns)),h.columns); plt.yticks(range(len(h.index)),h.index); plt.xlabel('C'); plt.ylabel('gamma'); plt.title(f'SVC CV Macro-F1 heatmap — {fs}'); plt.colorbar(label='Macro-F1'); plt.tight_layout(); plt.show()